# ER-CyRIS Siklus 2 — Pengukuran stabilitas deteksi per alert M2 di file1

Notebook ini **tidak** menyetel ambang baru atau mengubah gerbang M4. Inputnya adalah split terkunci file1 dan hasil JSON v2. Untuk setiap alert aktual pada calibration dan development, dalam keadaan bersih dan 10% missingness sintetis, skor `s_d_noise` dan `s_d_missing` mengukur fraksi dari 12 gangguan independen yang tetap melewati ambang detektor. Ini adalah **stabilitas keputusan di bawah dua protokol sintetis**, bukan probabilitas bahwa alert benar, kalibrasi, validitas risiko, atau ketahanan terhadap semua bentuk drift. Skor `s_d_min` adalah nilai terkecil dari kedua fraksi dan masih merupakan diagnostik; tidak masuk M4.

Protokol tetap: numeric noise pada matriks train-only transform dengan σ=0,01 **tanpa clipping fitur asli**, dan penghilangan tambahan 5% sel numerik pada input mentah yang ditransformasi ulang oleh preprocessor train-only. Setiap keluarga terdiri dari 12 benih tetap yang tidak bergantung pada label. Penghilangan 10% dari v2 memakai benih lain dan digunakan sebagai pemeriksaan *outcome* pada alert yang berasal dari kondisi bersih. Pada kondisi yang sudah terganggu 10%, penghilangan tambahan 5% berarti gangguan bersyarat pada observasi tersebut. Semua klaim dibatasi pada dataset UNSW file1. File2 tidak dibuka.

Output disimpan dalam folder baru: JSON ringkasan dan CSV terkompresi berisi satu baris per kandidat alert, termasuk split, kondisi, identitas baris sumber, label, margin `m_d`, stabilitas SHAP `r_x`, kelengkapan `q_e`, dan tiga diagnostik stabilitas deteksi. Karena agregat development v2 sudah diperiksa sebelum rancangan probe ini dibuat, AUC terhadap outcome 10% pada baris development adalah **diagnostik eksploratif**, sekalipun benih probe di sini berbeda. Arsip v1 dan v2 tidak ditimpa.


In [ ]:
import csv, hashlib, json, platform
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/ercyris_siklus2')
except ImportError:
    BASE=Path('.')
P1=BASE/'UNSW-NB15_1.csv'
LOCK_DIR=BASE/'unsw_locked_protocol_v1'
LOCK=LOCK_DIR/'unsw_lock_summary.json'
MANIFEST=LOCK_DIR/'unsw_locked_split_manifest.csv'
V2FILE=BASE/'unsw_framework_dev_v2'/'unsw_framework_development_v2.json'
OUT=BASE/'unsw_m2_stability_file1_v1'
OUTFILE=OUT/'unsw_m2_stability_summary.json'
EVIDFILE=OUT/'unsw_m2_stability_instances.csv.gz'
SEED=42; MISSING_RATE=.10; PROBE_SIGMA=.01; SHAP_MAX=300; TOPK=10
TAU_Q=.95; TAU_Q_SEVERE=.70; MIN_GATE_COVERAGE=.50
for path in (P1,LOCK,MANIFEST,V2FILE):
    if not path.is_file(): raise FileNotFoundError(f'Dibutuhkan: {path}')
if OUTFILE.exists() or EVIDFILE.exists():raise FileExistsError('Keluaran M2 sudah ada; STOP agar tidak menimpa.')
print('Output:',OUTFILE,'| File2 tidak akan dibuka.')

## 1. Verifikasi lock dan muat hanya baris file1

Identitas dua sumber tetap tercatat dalam lock, tetapi notebook ini membuka hanya file1. Kesamaan SHA sumber dan manifest, partisi, nomor baris, dan label diperiksa sebelum pelatihan. Bila ada ketidakcocokan, eksekusi berhenti.

In [ ]:
COLS=['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
      'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
      'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
      'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
      'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
      'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
      'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
      'ct_dst_src_ltm','attack_cat','label']
EXCLUDE={'label','attack_cat','srcip','dstip','stime','ltime'}
FEATURES=[c for c in COLS if c not in EXCLUDE]
CATS=['proto','state','service'];NUMS=[c for c in FEATURES if c not in CATS]
assert len(FEATURES)==43

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for piece in iter(lambda:f.read(4*1024*1024),b''):h.update(piece)
    return h.hexdigest()

lock=json.loads(LOCK.read_text(encoding='utf-8'))
if lock['status']!='FROZEN_SPLIT_NO_MODEL_NO_HOLDOUT_EVALUATION':
    raise ValueError('Status lock tidak sesuai; STOP.')
if sha256(P1)!=lock['sources']['file1']['sha256'] or sha256(MANIFEST)!=lock['manifest']['sha256']:
    raise ValueError('Berkas sumber atau manifest berubah sejak lock; STOP.')
manifest=pd.read_csv(MANIFEST,dtype={'feature_group_hash64':str})
if len(manifest)!=lock['manifest']['row_count']:
    raise ValueError('Jumlah baris manifest berubah; STOP.')
local=manifest.loc[manifest.source_file==P1.name].copy()
if set(local.partition)!={'train','calibration','development'} or len(local)!=30000:
    raise ValueError('Partisi file1 salah; STOP.')
if manifest.loc[manifest.partition=='holdout','source_file'].nunique()!=1:
    raise ValueError('Holdout bukan tepat satu file; STOP.')
if local.source_row_zero_based.duplicated().any():
    raise ValueError('Nomor baris file1 duplikat lintas partisi; STOP.')
with P1.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
    r=next(csv.reader(f))
if len(r)!=49 or r[-1].strip() not in ('0','1'):
    raise ValueError('Skema sumber tidak cocok; STOP.')
needed=set(local.source_row_zero_based.astype(int)); chunks=[]; offset=0
for chunk in pd.read_csv(P1,header=None,names=COLS,dtype=str,na_filter=False,chunksize=100000):
    positions=np.arange(offset,offset+len(chunk),dtype=np.int32)
    yes=np.isin(positions,list(needed))
    if yes.any():
        z=chunk.loc[yes].copy();z.index=positions[yes];chunks.append(z)
    offset+=len(chunk)
if offset!=lock['sources']['file1']['n']:raise ValueError('Jumlah baris file1 berbeda.')
raw=pd.concat(chunks).sort_index()
if len(raw)!=len(local):raise ValueError('Baris dalam manifest tidak seluruhnya ditemukan.')
labels=pd.to_numeric(raw['label'],errors='coerce')
if labels.isna().any() or not labels.isin([0,1]).all():raise ValueError('Label file1 tidak biner.')
lookup=local.set_index('source_row_zero_based').sort_index()
if not np.array_equal(labels.to_numpy(dtype=int),lookup['label'].to_numpy(dtype=int)):
    raise ValueError('Label manifest tidak sama dengan label sumber.')
for k in ('train','calibration','development'):
    print(k,lock['partitions'][k]['n'],'positif',lock['partitions'][k]['positives'])
print('Manifest tervalidasi. File2 belum dibaca.')

## 2. M1 kelengkapan dan transformasi M2 train only

Imputasi median, skala, dan penyandian kategori dipelajari dari `train` saja. Metadata provenance meliputi SHA sumber, nomor baris, dan rentang waktu; konteks aset eksternal tidak tersedia. Gangguan missingness yang disuntikkan ke calibration dan development didefinisikan terpisah; ia menguji reaksi aturan atas gangguan sintetis, **bukan** frekuensi gangguan nyata di organisasi.

In [ ]:
def prepare(frame):
    x=frame[FEATURES].copy()
    for c in NUMS:x[c]=pd.to_numeric(x[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in CATS:x[c]=x[c].astype(str).replace('', '__MISSING__')
    return x

def evidence_completeness(x):
    present=x[FEATURES].notna().copy()
    for c in CATS:present[c]&=x[c].astype(str).ne('__MISSING__')
    return present.mean(axis=1).to_numpy(dtype=float)

def synthetic_missing(frame, seed, frac=MISSING_RATE):
    v=frame.copy();rng=np.random.default_rng(seed)
    mask=rng.random((len(v),len(NUMS)))<frac
    for k,c in enumerate(NUMS):v.loc[mask[:,k],c]=np.nan
    return v

parts={name:lookup.loc[lookup.partition==name].index.to_numpy(dtype=np.int32)
       for name in ('train','calibration','development')}
xraw={name:prepare(raw.loc[ix]) for name,ix in parts.items()}
y={name:labels.loc[ix].to_numpy(dtype=np.int8) for name,ix in parts.items()}
xraw['cal_missing']=synthetic_missing(xraw['calibration'],SEED+1)
xraw['dev_missing']=synthetic_missing(xraw['development'],SEED+2)
y['cal_missing']=y['calibration'];y['dev_missing']=y['development']
q={name:evidence_completeness(xraw[name]) for name in xraw}
try:
    enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32)
except TypeError:
    enc=OneHotEncoder(handle_unknown='ignore',sparse=False,dtype=np.float32)
prep=ColumnTransformer([
    ('numeric',Pipeline([('median',SimpleImputer(strategy='median')),
                         ('scale',MinMaxScaler())]),NUMS),
    ('categorical',enc,CATS)
],remainder='drop',sparse_threshold=0)
X={'train':np.asarray(prep.fit_transform(xraw['train']),dtype=np.float32)}
for name in xraw:
    if name!='train':X[name]=np.asarray(prep.transform(xraw[name]),dtype=np.float32)
if any(not np.isfinite(v).all() for v in X.values()):raise ValueError('Vektor fitur nonhingga.')
if len({v.shape[1] for v in X.values()})!=1:raise ValueError('Dimensi antarpartisi berbeda.')
print('Fitur hasil train-only transform:',X['train'].shape[1])
print('q_e completeness clean calibration:',float(q['calibration'].mean()),
      'missingness synthetic:',float(q['cal_missing'].mean()))

## 3. Definisi ukuran deteksi dan penjelasan

Metrik `m_d` dan `r_x` harus identik dengan cara pengukuran v2 agar setiap baris dapat diaudit. Skor stabilitas M2 didefinisikan dengan intervensi pada fitur dan **tidak** memakai label. Jika alert tidak lolos pada suatu gangguan, hasilnya nol untuk percobaan tersebut. Komponen noise dan missingness dilaporkan terpisah sebelum nilai minimum. `s_d_min` tidak dinamai probabilitas benar atau `r_d` terkalibrasi. Fungsi gerbang dua domain dipakai hanya untuk memeriksa bahwa bukti bersih tetap sejalan dengan v2; tidak ada ambang yang dipilih dari hasil baru. Probes M2 tidak memotong nilai di luar rentang skala latih; kolom audit mencatat jumlah fitur demikian. Probe SHAP M3 tetap memakai definisi v2 agar perbandingan dapat direproduksi, dan kemungkinan efek clipping dilaporkan sebagai batasnya.


In [ ]:
def select_detector_threshold(y_true,p):
    candidates=np.linspace(.05,.95,19)
    scored=[]
    for t in candidates:
        pred=(p>=t).astype(int)
        tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
        f=f1_score(y_true,pred,zero_division=0)
        far=float(fp/(fp+tn)) if fp+tn else 1.
        scored.append((float(f),-far,-abs(float(t)-.5),float(t)))
    return max(scored)[3]

def margin_d(probability,detector_threshold):
    p=np.clip(np.asarray(probability,dtype=float),0.,1.)
    return np.clip((p-detector_threshold)/max(1.-detector_threshold,1e-12),0.,1.)

def add_numeric_probe(x,seed,sigma=PROBE_SIGMA):
    v=np.asarray(x,dtype=np.float32).copy();rng=np.random.default_rng(seed)
    v[:,:len(NUMS)]=np.clip(v[:,:len(NUMS)]+rng.normal(0,sigma,(len(v),len(NUMS))),0,1)
    return v

def tree_shap_positive(model,x):
    try:
        v=shap.TreeExplainer(model).shap_values(x,check_additivity=False)
    except TypeError:
        v=shap.TreeExplainer(model).shap_values(x)
    if isinstance(v,list):v=v[1]
    v=np.asarray(v,dtype=float)
    if v.ndim==3:
        if v.shape[-1]!=2:raise ValueError(f'SHAP expected binary class, got {v.shape}')
        v=v[:,:,1]
    if v.shape!=x.shape or not np.isfinite(v).all():
        raise ValueError(f'SHAP gagal atau bentuknya berbeda: {v.shape} vs {x.shape}')
    return v

def explanation_stability(clean,probe,k=TOPK):
    a=np.asarray(clean,dtype=float);b=np.asarray(probe,dtype=float)
    if a.shape!=b.shape or a.ndim!=2:raise ValueError('SHAP dua kondisi harus seukuran.')
    k=min(int(k),a.shape[1]);
    ac=np.abs(a);bc=np.abs(b)
    ta=np.argsort(-ac,axis=1)[:,:k];tb=np.argsort(-bc,axis=1)[:,:k]
    jac=[];sign=[]
    for i,(left,right) in enumerate(zip(ta,tb)):
        u=set(left)|set(right);jac.append(len(set(left)&set(right))/len(u))
        pos=sorted(u);sign.append(float(np.mean(np.sign(a[i,pos])==np.sign(b[i,pos]))))
    den=ac.mean(axis=1)+1e-12
    mag=np.clip(1-np.abs(ac-bc).mean(axis=1)/den,0,1)
    components=np.column_stack([jac,mag,sign]);return np.clip(components,0,1).mean(axis=1)

def gate2(m_d,r_x,tau_d,tau_x):
    a=np.asarray(m_d)>=tau_d;b=np.asarray(r_x)>=tau_x
    return np.where(a&b,'accept',np.where(a|b,'warn','abstain'))


## 4. Rekonstruksi detektor dan pembuktian identitas v2

Notebook memuat ulang file1 dan merekonstruksi pelatihan dua model. Sebelum menghitung ukuran M2, ia harus menemukan SHA, split, ambang detektor, matriks kebingungan, F1, dan AP yang sama dengan JSON v2. Jika salah satu tidak cocok, eksperimen berhenti; statistik dari dua protokol yang berbeda tidak dicampur.


In [ ]:
models={
 'XGBoost':XGBClassifier(random_state=SEED,n_jobs=-1,eval_metric='logloss',verbosity=0,tree_method='hist'),
 'RandomForest':RandomForestClassifier(random_state=SEED,n_jobs=-1),
}
trained={};thresholds={};probs={};detector_scores={}
for name,model in models.items():
    model.fit(X['train'],y['train']);trained[name]=model
    pcal=model.predict_proba(X['calibration'])[:,1]
    threshold=select_detector_threshold(y['calibration'],pcal);thresholds[name]=threshold
    probs[name]={'calibration':pcal}
    detector_scores[name]={}
    for split in ('calibration','development','cal_missing','dev_missing'):
        prob=pcal if split=='calibration' else model.predict_proba(X[split])[:,1]
        probs[name][split]=prob
        pred=prob>=threshold
        tn,fp,fn,tp=confusion_matrix(y[split],pred,labels=[0,1]).ravel()
        detector_scores[name][split]={
            'n':int(len(pred)),'threshold':threshold,
            'F1':float(f1_score(y[split],pred,zero_division=0)),
            'AP':float(average_precision_score(y[split],prob)),
            'FAR':float(fp/(tn+fp)) if tn+fp else None,
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        }
    print(name,'detector threshold calibration=',threshold,
          'F1 clean development=',round(detector_scores[name]['development']['F1'],4))
reference=json.loads(V2FILE.read_text(encoding='utf-8'))
if reference['status']!='DEVELOPMENT_ONLY_CORRECTED_V2_FILE2_UNTOUCHED':
    raise ValueError('JSON acuan bukan hasil v2 yang diharapkan; STOP.')
if reference['lock_manifest_sha256']!=lock['manifest']['sha256'] or reference['source_file1_sha256']!=lock['sources']['file1']['sha256']:
    raise ValueError('Protokol sumber/manifest berbeda dari v2; STOP.')
for name in trained:
    if abs(thresholds[name]-reference['detector_thresholds_from_calibration'][name])>1e-12:
        raise ValueError(f'Ambang detektor {name} berubah; STOP.')
    for split in ('calibration','development','cal_missing','dev_missing'):
        current=detector_scores[name][split]; earlier=reference['detector_results'][name][split]
        for key in ('n','TN','FP','FN','TP'):
            if current[key]!=earlier[key]:raise ValueError(f'{name} {split}: {key} berbeda dari v2; STOP.')
        for key in ('F1','AP','FAR'):
            if abs(current[key]-earlier[key])>1e-10:raise ValueError(f'{name} {split}: {key} berbeda dari v2; STOP.')
print('Rekonstruksi hasil detektor v2 lolos; file2 belum dibaca.')


## 5. Hasil per alert dan diagnostik yang tidak dipakai menyetel gerbang

Seluruh kandidat alert aktual dihitung tanpa sampling, sepanjang jumlah tiap kondisi tidak melebihi batas SHAP v2. Benih gangguan hanya fungsi dari model, split, kondisi, jenis probe, nomor ulangan, dan seed 42. Untuk alert bersih, kolom `retained_under_v2_10pct_missing` menunjukkan apakah alert yang sama bertahan pada satu gangguan 10% dengan benih berbeda dari probe di sini. ROC AUC stabilitas hanya merupakan diagnostik ranking eksploratif terhadap label tersebut; development telah dilihat dalam agregat v2, sehingga bukan confirmatory holdout. Tidak membuktikan kalibrasi, sebab, atau probabilitas kebenaran alert. Nilai q_e dan label tidak masuk komputasi stabilitas.


In [ ]:
NOISE_PROBES=12;MISSING_PROBES=12;PROBE_MISSING_RATE=.05;PROBE_NOISE_SIGMA=.01

def jitter_numeric_without_clipping(xmat,seed,sigma):
    out=np.asarray(xmat,dtype=np.float32).copy()
    noise=np.random.default_rng(seed).normal(0,sigma,(len(out),len(NUMS))).astype(np.float32)
    out[:,:len(NUMS)]+=noise
    return out

def stable_seed(*labels):
    payload='|'.join(map(str,(SEED,)+labels)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(payload).digest()[:8],'little')

def detection_stability(model,name,split,condition,xmat,xframe,threshold):
    noise_hits=np.zeros(len(xmat),dtype=np.float64)
    missing_hits=np.zeros(len(xmat),dtype=np.float64)
    for rep in range(NOISE_PROBES):
        probe=jitter_numeric_without_clipping(xmat,stable_seed(name,split,condition,'noise',rep),sigma=PROBE_NOISE_SIGMA)
        noise_hits += model.predict_proba(probe)[:,1]>=threshold
    for rep in range(MISSING_PROBES):
        raw_probe=synthetic_missing(xframe,stable_seed(name,split,condition,'missing',rep),frac=PROBE_MISSING_RATE)
        transformed=np.asarray(prep.transform(raw_probe),dtype=np.float32)
        missing_hits += model.predict_proba(transformed)[:,1]>=threshold
    out_noise=noise_hits/NOISE_PROBES;out_missing=missing_hits/MISSING_PROBES
    if not np.isfinite(out_noise).all() or not np.isfinite(out_missing).all():
        raise ValueError('Skor stabilitas tidak valid; STOP.')
    return out_noise,out_missing,np.minimum(out_noise,out_missing)

def ranking_auc(target,score):
    target=np.asarray(target,dtype=int)
    if np.unique(target).size<2:return None
    return float(roc_auc_score(target,np.asarray(score,dtype=float)))

records=[];audits={};insights={}
for name,model in trained.items():
    audits[name]={};insights[name]={}
    frozen=reference['gate_thresholds_from_calibration'][name]
    threshold=thresholds[name]
    for split,condition,key in [('calibration','clean','calibration'),
                                ('calibration','synthetic_missing','cal_missing'),
                                ('development','clean','development'),
                                ('development','synthetic_missing','dev_missing')]:
        candidate=np.flatnonzero(probs[name][key]>=threshold)
        expected=reference['candidate_sampling_audit'][name]['dynamic'][split+'/'+condition]
        if len(candidate)!=expected['detector_positive_before_cap'] or expected['sampled'] or len(candidate)>SHAP_MAX:
            raise ValueError(f'{name} {key}: kandidat berubah atau sampling diperlukan; STOP.')
        xmat=X[key][candidate]
        xframe=xraw[key].iloc[candidate]
        rn,rm,rmin=detection_stability(model,name,split,condition,xmat,xframe,threshold)
        # Penjelasan dihitung ulang hanya untuk audit keselarasan gerbang dua domain v2.
        shap_base=tree_shap_positive(model,xmat)
        shap_probe=tree_shap_positive(model,add_numeric_probe(xmat,SEED+5))
        rx=explanation_stability(shap_base,shap_probe)
        outside_numeric=((xmat[:,:len(NUMS)]<0)|(xmat[:,:len(NUMS)]>1)).sum(axis=1)
        md=margin_d(probs[name][key][candidate],threshold)
        action=gate2(md,rx,frozen['tau_d'],frozen['tau_x'])
        true_y=y[split][candidate]
        predicted_truth=(action=='accept')
        count={'candidate_alerts':int(len(candidate)),
               'accept':int(predicted_truth.sum()),
               'warn':int((action=='warn').sum()),
               'abstain':int((action=='abstain').sum()),
               'TP_accept':int((predicted_truth&(true_y==1)).sum()),
               'FP_accept':int((predicted_truth&(true_y==0)).sum())}
        earlier=reference['dynamic_gate_results'][name][split+'/'+condition]['two_domain_gate']
        for k,value in count.items():
            if value!=earlier[k]:raise ValueError(f'Gerbang dua domain {name} {key} {k}: {value} vs {earlier[k]}; STOP.')
        audit_key=split+'/'+condition;audits[name][audit_key]=count
        retained=(probs[name]['cal_missing' if split=='calibration' else 'dev_missing'][candidate]>=threshold).astype(int) if condition=='clean' else None
        frame=pd.DataFrame({
          'lock_protocol_id':lock['protocol_id'],
          'source_file':P1.name,
          'source_row_zero_based':parts[split][candidate],
          'model':name,'split':split,'condition':condition,
          'y_true':true_y.astype(int),
          'detector_probability':probs[name][key][candidate],
          'detector_threshold':threshold,
          'q_completeness':q[key][candidate],
          'm_d_margin':md,'r_x_local_shap':rx,
          'numeric_values_outside_train_scaler_range':outside_numeric,
          's_d_noise':rn,'s_d_missing':rm,'s_d_min':rmin,
          'gate2_action':action,
          'retained_under_v2_10pct_missing':retained if retained is not None else np.full(len(candidate),np.nan),
        })
        records.append(frame)
        insight={'n':int(len(frame)),'true_alerts':int((true_y==1).sum()),
                 'false_alerts':int((true_y==0).sum()),
                 'mean_s_d_noise':float(rn.mean()),
                 'mean_s_d_missing':float(rm.mean()),
                 'mean_s_d_min':float(rmin.mean()),
                 'auc_s_d_min_for_alert_correctness':ranking_auc(true_y,rmin),
                 'auc_m_d_for_alert_correctness':ranking_auc(true_y,md),
                 'count_s_d_min_below_0p5':int((rmin<.5).sum()),
                 'count_s_d_min_0p5_to_below_0p9':int(((rmin>=.5)&(rmin<.9)).sum()),
                 'count_s_d_min_at_least_0p9':int((rmin>=.9).sum()),
                 'alerts_with_numeric_outside_train_scaler_range':int((outside_numeric>0).sum()),
                }
        if retained is not None:
            insight.update({'clean_alerts_lost_under_independent_10pct_missing':int((1-retained).sum()),
                            'auc_s_d_min_for_alert_retention_under_10pct_missing':ranking_auc(retained,rmin),
                            'auc_s_d_missing_for_alert_retention_under_10pct_missing':ranking_auc(retained,rm),
                            'auc_m_d_for_alert_retention_under_10pct_missing':ranking_auc(retained,md)})
        insights[name][audit_key]=insight
        print(name,audit_key,'candidates=',len(frame),'mean stability=',round(float(rmin.mean()),3),
              '10% retention AUC=',insight.get('auc_s_d_min_for_alert_retention_under_10pct_missing'))

instances=pd.concat(records,ignore_index=True)
if instances.duplicated(['model','split','condition','source_row_zero_based']).any():
    raise ValueError('Duplikat identitas baris per model dan kondisi; STOP.')
print('Semua bukti per alert siap:',len(instances),'baris; holdout file2 tidak dibuka.')


## 6. Arsipkan diagnostik, bukan klaim validasi r_d

CSV gzip menyimpan skor per baris agar analisis lanjutan tidak perlu melatih ulang detektor hanya untuk membaca nilai yang telah dihasilkan. JSON berisi SHA berkas CSV dan diagnostik agregat; keduanya harus disimpan bersama. Korelasi atau AUC yang tinggi hanya berlaku pada protokol gangguan terdefinisi. Tidak ada threshold baru M4 atau q_e yang dipilih pada langkah ini. File2 tetap tertutup.


In [ ]:
if OUTFILE.exists() or EVIDFILE.exists():
    raise FileExistsError('Salah satu keluaran M2 sudah ada; STOP sebelum menimpa.')
OUT.mkdir(parents=True,exist_ok=True)
instances.to_csv(EVIDFILE,index=False,compression={'method':'gzip','mtime':0})
result={
  'status':'UNSW_FILE1_M2_STABILITY_DIAGNOSTIC_ONLY_FILE2_UNTOUCHED',
  'source_version':'v2 model, split, detector threshold and clean gate 2 reproduced',
  'lock_protocol_id':lock['protocol_id'],
  'lock_manifest_sha256':lock['manifest']['sha256'],
  'source_file1_sha256':lock['sources']['file1']['sha256'],
  'v2_reference_sha256':sha256(V2FILE),
  'per_instance_csv_sha256':sha256(EVIDFILE),
  'per_instance_csv_name':EVIDFILE.name,
  'per_instance_rows':int(len(instances)),
  'config':{'seed':SEED,'noise_probe_count':NOISE_PROBES,
            'noise_sigma_on_numeric_transformed':PROBE_NOISE_SIGMA,
            'missing_probe_count':MISSING_PROBES,
            'missing_fraction_on_raw_numerics':PROBE_MISSING_RATE,
            'independent_existing_10pct_missing_seed_calibration':SEED+1,
            'independent_existing_10pct_missing_seed_development':SEED+2,
            'score_definition':'s_d_noise and s_d_missing are alert retention fractions; s_d_min is their minimum',
            'gate_updated':False,'q_e_veto_used':False,'holdout_accessed':False},
  'detector_thresholds':thresholds,
  'gate_thresholds_reused_v2':reference['gate_thresholds_from_calibration'],
  'gate2_reproduction_audit':audits,
  'diagnostics':insights,
  'limitations':['Synthetic fixed-seed perturbations, not all real missingness or drift',
                 'Noise robustness score does not clip transformed features; v2 SHAP probe still clips them, and out-of-range numeric inputs are audited per row',
                 'Alert stability is not correctness probability or calibration',
                 '10pct outcome on clean alerts is one independent random realization',
                 'Development v2 aggregate outcomes were inspected before this score design; AUC here is exploratory, not independent confirmation',
                 'Scores computed only on actual alerts; selection bias prohibits population inference',
                 'No new M4 threshold fitted; q_e remains diagnostic',
                 'file2 holdout not accessed and no risk/response M5/M6 tested']}
OUTFILE.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('Output diagnostik:',OUTFILE)
print('Bukti per alert:',EVIDFILE,'SHA256:',result['per_instance_csv_sha256'])
print('FILE2 TETAP TERTUTUP.')
